In [ ]:
import os, sys, json
sys.path.append('..')
from dotenv import load_dotenv
import anthropic

load_dotenv()
client = anthropic.Anthropic()
MODEL = "claude-sonnet-4-5"

## 1. JSON con System Prompt

In [ ]:
def extract_json(text: str) -> dict:
    """Extrae y parsea JSON de la respuesta de Claude."""
    if '```json' in text:
        start = text.index('```json') + 7
        end = text.index('```', start)
        text = text[start:end].strip()
    elif '```' in text:
        start = text.index('```') + 3
        end = text.index('```', start)
        text = text[start:end].strip()
    return json.loads(text.strip())

system_json = """
Responde UNICAMENTE con JSON valido, sin texto adicional ni bloques de codigo.
"""

user_msg = """
Extrae la informacion del siguiente texto en JSON con campos: 
nombre, edad, profesion, ciudad.

Texto: "Maria Garcia tiene 34 años, trabaja como ingeniera de software 
y vive en Barcelona desde hace 5 años."
"""

resp = client.messages.create(
    model=MODEL, max_tokens=300,
    system=system_json,
    messages=[{"role": "user", "content": user_msg}]
)

raw = resp.content[0].text
print("Respuesta raw:", raw)

data = json.loads(raw)
print("\nJSON parseado:")
print(json.dumps(data, ensure_ascii=False, indent=2))


## 2. Prefill para Forzar JSON

In [ ]:
# El prefill inicia la respuesta del asistente con '{'
# Esto garantiza que Claude continúe con JSON

resp = client.messages.create(
    model=MODEL, max_tokens=500,
    messages=[
        {
            "role": "user",
            "content": """Genera una lista de 3 productos tecnológicos con sus características.
            Formato JSON: {"productos": [{"nombre": ..., "precio": ..., "categoria": ...}]}"""
        },
        {
            "role": "assistant",
            "content": "{"  # Prefill: iniciamos la respuesta del asistente
        }
    ]
)

# La respuesta no incluye el '{' inicial, lo añadimos
json_str = "{" + resp.content[0].text
print("JSON completo:")
data = json.loads(json_str)
print(json.dumps(data, ensure_ascii=False, indent=2))

## 3. Ejercicio: Pipeline de Extracción de Datos

In [ ]:
reviews = [
    "El iPhone 15 es increible, la camara es perfecta. Lo compre por $999. 5 estrellas",
    "Decepcionante. La Samsung Galaxy no duro ni un año. Pague $750. Solo 2 estrellas.",
    "El MacBook Pro vale cada centavo. Potente, elegante. $1299. Me encanta. 5/5",
]

system_extractor = """
Eres un extractor de datos. Para cada resena, devuelve JSON con:
{
  "producto": string,
  "sentimiento": "positivo" | "negativo" | "neutral",
  "precio": number,
  "puntuacion": number  (en escala 1-5)
}
Solo JSON, sin texto adicional.
"""

resultados = []
for review in reviews:
    resp = client.messages.create(
        model=MODEL, max_tokens=200,
        system=system_extractor,
        messages=[{"role": "user", "content": review}]
    )
    try:
        data = json.loads(resp.content[0].text)
        resultados.append(data)
        print(f"OK: {data}")
    except json.JSONDecodeError as e:
        print(f"Error parseando: {resp.content[0].text}")

print("\nResumen:")
print(f"  Positivos: {sum(1 for r in resultados if r.get('sentimiento') == 'positivo')}")
print(f"  Negativos: {sum(1 for r in resultados if r.get('sentimiento') == 'negativo')}")
precios = [r.get('precio', 0) for r in resultados if r.get('precio')]
print(f"  Precio promedio: ${sum(precios)/len(precios):.0f}")
